<a href="https://colab.research.google.com/github/rajeshsharma38/neural-network-learning/blob/main/MINST_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd

In [2]:
import torch.nn.functional as F

In [3]:
df = pd.read_csv('mnist_train.csv', header=None)
df

,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,784
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59997,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59998,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
df.shape
Ytr = torch.tensor(df.iloc[:, 0].values, dtype=torch.long)
Xtr = torch.tensor(df.iloc[:, 1:].values, dtype=torch.float32)
Xtr.shape, Ytr.shape, Xtr.dtype, Ytr.dtype

(torch.Size([60000, 784]), torch.Size([60000]), torch.float32, torch.int64)

In [25]:
#check kamming_normal initialisation
w1 = torch.randn(784, 100) / 784**0.5
b1 = torch.randn(100) * 0.01
w2 = torch.randn(100, 10) / 100**0.5
b2 = torch.randn(10) * 0

parameters = [w1, b1, w2, b2]

for p in parameters:
  p.requires_grad = True


In [26]:
print(sum(p.numel() for p in parameters))

79510


In [27]:
X_mean = Xtr.mean(0, keepdim=True)
X_std = Xtr.std(0, keepdim=True)
eps = 0.01

In [28]:
#forward pass
batch_size = 50
batch_norm = torch.nn.BatchNorm1d(num_features=100, momentum=0.01)
for i in range(40000): # Reduced iterations for faster debugging
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb = Xtr[ix]
  Yb = Ytr[ix]

  # Xb_mean = Xb.mean(0, keepdim=True)
  # Xb_std = Xb.std(0, keepdim=True)

  # X_mean = 0.998 * X_mean + 0.002 * Xb_mean
  # X_std = 0.998 * X_std + 0.002 * Xb_std

  # Xb = (Xb - Xb_mean) / (Xb_std + eps)

  hpreact = batch_norm(Xb @ w1 + b1)

  # Detach, convert to numpy, flatten, and then remove NaNs before plotting
  # import numpy as np # Import numpy if not already imported
  # hpreact_flat = hpreact.detach().numpy().flatten()
  # hpreact_flat = hpreact_flat[~np.isnan(hpreact_flat)] # Filter out NaN values

  # if hpreact_flat.size > 0: # Only plot if there's data after removing NaNs
  #   plt.hist(hpreact_flat, bins=200) # Plot a histogram of flattened hpreact values
  #   plt.title('Distribution of hpreact values (NaNs removed)') # Add a title
  #   plt.xlabel('Value') # Add x-axis label
  #   plt.ylabel('Frequency') # Add y-axis label
  #   plt.show() # Ensure the plot is displayed
  # else:
  #   print("No valid data to plot after removing NaNs from hpreact.")

  # break # The break statement is currently preventing the training loop from running

  h = torch.tanh(hpreact)

  logits = h @ w2 + b2
  loss = F.cross_entropy(logits, Yb)

  #print(loss.item())

  for p in parameters:
    p.grad = None

  loss.backward()

  lr = 0.1 if i < 20000 else 0.01

  for p in parameters:
    p.data += -lr * p.grad

  if i%100 == 0: # Print loss more frequently with fewer iterations
    print(loss.item())

2.4225246906280518
0.37654948234558105
0.4412543773651123
0.29537832736968994
0.2948688566684723
0.310995489358902
0.25453412532806396
0.11286388337612152
0.3742651045322418
0.24327674508094788
0.3354507088661194
0.2046009600162506
0.1581667959690094
0.10395839810371399
0.14498096704483032
0.1113545149564743
0.3196244239807129
0.2409040778875351
0.25229912996292114
0.1751246452331543
0.2186521738767624
0.16431407630443573
0.1291307955980301
0.3311234414577484
0.09655830264091492
0.09984023869037628
0.10839775204658508
0.04552633315324783
0.15278375148773193
0.16205407679080963
0.2103620320558548
0.3452567756175995
0.20875155925750732
0.10578089952468872
0.4113472104072571
0.18177656829357147
0.29876935482025146
0.17646217346191406
0.3858147859573364
0.146447092294693
0.32623210549354553
0.15803533792495728
0.09595299512147903
0.09423664212226868
0.05339110270142555
0.28942909836769104
0.05826471745967865
0.28512972593307495
0.1273522824048996
0.1567523032426834
0.14457088708877563
0.16

Now that you've trained the model, let's use it to predict digits on a small portion of the test set (`mnist_test.csv`). We'll take the first 10 examples to quickly see how it performs.

In [29]:
# Load the test dataset (mnist_test.csv)
df_test = pd.read_csv('mnist_test.csv', header=None)

# Take only the first 10 examples for prediction
X_test = torch.tensor(df_test.iloc[:, 1:].values, dtype=torch.float32)
Y_test = torch.tensor(df_test.iloc[:, 0].values, dtype=torch.long)

print(f"Test data shape (features): {X_test.shape}")
print(f"Test data shape (labels): {Y_test.shape}")

Test data shape (features): torch.Size([10000, 784])
Test data shape (labels): torch.Size([10000])


In [34]:
# Perform forward pass on the test data using the trained weights (w1, b1, w2, b2)
hpreact_test = X_test @ w1 + b1
h_test = torch.tanh(batch_norm(hpreact_test))
logits_test = h_test @ w2 + b2

# Get the predicted class by finding the index of the maximum logit value
predictions = torch.argmax(logits_test, dim=1)

match_count = 0

print("--- Predictions vs Actual Labels (First 10 Examples) ---")
for i in range(len(predictions)):
    if predictions[i].item() == Y_test[i].item():
      match_count += 1
print(f"Matched: {match_count} \n Unmatched: {len(predictions) - match_count}")
batch_norm.train()

--- Predictions vs Actual Labels (First 10 Examples) ---
Matched: 9721 
 Unmatched: 279


BatchNorm1d(100, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)

In [22]:
#X_tr1 = batch_norm(Xtr)

In [35]:
batch_norm.eval() # Set batch_norm to evaluation mode

hpreact_test = Xtr @ w1 + b1
h_test = torch.tanh(batch_norm(hpreact_test))
logits_test = h_test @ w2 + b2

# Get the predicted class by finding the index of the maximum logit value
predictions = torch.argmax(logits_test, dim=1)

match_count = 0

print("--- Predictions vs Actual Labels ---")
for i in range(len(predictions)):
    if predictions[i].item() == Ytr[i].item():
      match_count += 1
print(f"Matched: {match_count} \n Unmatched: {len(predictions) - match_count}")

batch_norm.train() # Set batch_norm back to training mode if you plan further training

--- Predictions vs Actual Labels ---
Matched: 59571 
 Unmatched: 429


BatchNorm1d(100, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)